# Step 1: Import helpful libraries

In [ ]:
# Regular imports
import numpy as np
import pandas as pd
import sklearn as sk
import lightgbm as lhgbm

#For Visualizations
import pandas_profiling as pp
import matplotlib as mpl
import seaborn as sns

#Ensemble Models
import xgboost as xgb

import matplotlib.pyplot as plt
%matplotlib inline
sns.set()
plt.style.use('classic')
pd.set_option('max_columns', None)

from sklearn.linear_model import *


#To Split Data 
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, GridSearchCV, cross_val_score, KFold

#To Preprocess Data 
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder, FunctionTransformer, OneHotEncoder, Normalizer
from sklearn.impute import SimpleImputer
from imblearn import FunctionSampler


#To Pipeline the process 
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
#from lineartree import  LinearTreeRegressor, LinearBoostRegressor
from sklearn.pipeline import Pipeline

# used in Utilities/Functions section
from scipy import stats

## Import Models
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from mlxtend.regressor import StackingCVRegressor


# Import metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, r2_score
from xgboost import plot_importance


# variables
target_feature = 'target' 
random_seed    = 42
max_rows_per_class  = 100 ## use high number (300000) to get all data 
full_run ='Y'  

In [ ]:
#Python libraries and their versions used for this problem
print('SciKit Learn:',sk.__version__)
print('Pandas:',pd.__version__)
print('Numpy:',np.__version__)
print('Seaborn:',sns.__version__)
print('MatPlot Library:', mpl.__version__)
print('XG Boost:',xgb.__version__)
print('Pandas Profiling:', pp.__version__)
print('LightGBM:', lhgbm.__version__)


In [ ]:
#Define Utilities/Functions

# Function to show essential info about Dataset
def ShowEssentialInfo(df):
    # Check No of rows & columns
    print("\n",'*** Shape:',df.shape)

    # Check data
    print("\n",'*** Data:',df.head(10))
    
    # Check Info about Object types of data
    print("\n",'*** Info:')
    df.info()

    #Count missing values 
    print("\n",'*** Missing values:')
    print(df.isnull().sum())

    #Data Statistics
    print("\n",'*** Data Statistics:')
    print(df.describe(include='all'))
    
    #Data Skew
    #print("\n",'*** Skewness:')
    #print('#** skewness is a degree of asymmetry observed in a probability distribution that deviates from the symmetrical normal distribution (bell curve) in a given set of data')
    #print('#** If the skewness is between -0.5 & 0.5, the data are nearly symmetrical.')
    #print('#** If the skewness is between -1 & -0.5 (negative skewed) or between 0.5 & 1(positive skewed), the data are slightly skewed.')
    #print('#** If the skewness is lower than -1 (negative skewed) or greater than 1 (positive skewed), the data are extremely skewed.')
    #print(df.skew(axis=1))
 
    #Data Kurtosis
    #print("\n",'*** Kurtosis:')
    #print('#** Kurtosis quantifies shape of the distribution and the degree of presence of outliers in the distribution.')
    #print('# High kurtosis in a data set is an indicator that data has heavy outliers.')
    #print('# Low kurtosis in a data set is an indicator that data has lack of outliers.')
    #print('# If kurtosis value + means pointy and — means flat.')    
    #print(df.kurt(axis=1))
 
    

def treatoutliers(df=None, columns=None, factor=1.5, method='IQR', treatment='cap'):

    for column in columns:
        if method == 'STD':
            permissable_std = factor * df[column].std()
            col_mean = df[column].mean()
            floor, ceil = col_mean - permissable_std, col_mean + permissable_std
        elif method == 'IQR':
            Q1 = df[column].quantile(0.25)
            Q3 = df[column].quantile(0.75)
            IQR = Q3 - Q1
            floor, ceil = Q1 - factor * IQR, Q3 + factor * IQR
#         print(floor, ceil)
        if treatment == 'remove':
            print(treatment, column)
            df = df[(df[column] >= floor) & (df[column] <= ceil)]
        elif treatment == 'cap':
            print(treatment, column)
            df[column] = df[column].clip(floor, ceil)

    return df
    
def get_sample_dataset(df, categorical_features, max_rows_per_class=1000):
    rows = []
    df_sub = pd.DataFrame()
    for x in categorical_features:
        for idx,name in enumerate(df[x].value_counts().index.tolist()):
            nrows = df[x].value_counts()[idx]
            nsample = min(nrows, max_rows_per_class)
            data = df.loc[df[x] == name].sample(n=nsample, random_state=random_seed)
            #print(data.info())
            df_sub = df_sub.append(data)
    return df_sub


def log_transform(x):
    return np.log(x + 1)

def exp_transform(x):
    return np.exp(x)



# Step 2: Load the data

Next, we'll load the training and test data.  

We set `index_col=0` in the code cell below to use the `id` column to index the DataFrame.

In [ ]:
# Load the training data
X_full = pd.read_csv("../input/30-days-of-ml/train.csv", index_col=0)
X_test_full = pd.read_csv("../input/30-days-of-ml/test.csv", index_col=0)

X_all = pd.concat([X_full,X_test_full]) 


In [ ]:
## Check for Data types & Missing data
ShowEssentialInfo(X_full)

No Null values in Data

In [ ]:
ShowEssentialInfo(X_test_full)

In [ ]:
# Identify Numeric / Categorical features this would be used to transform features later 

# "Cardinality" means the number of unique values in a column
# Select categorical columns with relatively low cardinality (convenient but arbitrary)
categorical_features  = [cname for cname in X_full.columns if
                    X_full[cname].dtype in ["object"]]

# Select numerical columns
numeric_features  = [cname for cname in X_full.columns if 
                X_full[cname].dtype in ['int64', 'float64']
                 ]

# Keep selected columns only
my_features = categorical_features + numeric_features

print('categorical_features({}):'.format(len(categorical_features)),categorical_features)
print('numeric_features({}):'.format(len(numeric_features)), numeric_features)
print('my_features:({})'.format(len(my_features)), my_features)


In [ ]:
# Identify Numeric / Categorical features this would be used to transform features later 

# "Cardinality" means the number of unique values in a column
# Select categorical columns with relatively low cardinality (convenient but arbitrary)
categorical_features  = [cname for cname in X_full.columns if
                    X_full[cname].dtype == "object"]

# Select numerical columns
numeric_features  = [cname for cname in X_full.columns if 
                X_full[cname].dtype in ['int64', 'float64']
                 ]

# Keep selected columns only
my_features = categorical_features + numeric_features

print('categorical_features({}):'.format(len(categorical_features)),categorical_features)
print('numeric_features({}):'.format(len(numeric_features)), numeric_features)
print('my_features:({})'.format(len(my_features)), my_features)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

#print(X_random_subset['cat9'].value_counts(()).index.tolist())

## Feature Selection using variance_inflation_factor
def cal_vif(X, thresh=6):
    output = pd.DataFrame()
    k =X_vif.shape[1]
    vif = [variance_inflation_factor(X_vif.values, i) for i in range(k)]
    for i in range(1,k):
        print('Iteration No ', i)
        print(vif)
        a = int(np.argmax(vif))
        if(vif[a]<=thresh):
            print('break')
            break
        if(i==1):
            print('i=1', i)
            output=X_vif.drop(X_vif.columns[a], axis=1)  
        elif(i>1):
            print('i>1', i)
            output=output.drop(output.columns[a], axis=1)
        l = len(output.columns)    
        vif=[variance_inflation_factor(output.values, j) for j in range(l)]
    return(output)     


# creating dummies for categorical_features

# the independent variables set
X_vif=X_all.copy()
y_vif=X_vif[target_feature]
X_vif=X_vif.drop(target_feature,1)

#{'A':0, 'B':1, 'C':2, 'D':3, 'E':5, 'F':6,'G':7,'H':8,'I':9,'J':10,'K':11,'L':12,'M':13,'N':14,'O':15}
X_vif[categorical_features] = X_vif[categorical_features] = X_vif[categorical_features].applymap(lambda x: ord(x)-65)

#selected_features = pd.DataFrame()
selected_features = cal_vif(X_vif)
#selected_features = X_full.head(5)
selected_features.head()

In [ ]:
selected_features['target'] = y_vif

In [ ]:
exclude_columns = set(list(X_full.columns)) - set(list(selected_features.columns))
print(exclude_columns)


In [ ]:
ShowEssentialInfo(selected_features)

In [ ]:
## Correlations
correlations = X_full.corr()
f, ax = plt.subplots(figsize=(12, 12))
sns.heatmap(correlations, square=True, cbar=True, annot=True, vmax=.9);

In [ ]:
#Correlation with output variable
cor_target = abs(correlations[target_feature])
#Selecting highly correlated features
relevant_features = cor_target[cor_target>0]
relevant_features

All features are weakly correlated to target feature

In [ ]:
## Data Distribution of numeric features 
X_full[numeric_features].hist(bins=100, figsize=(24,12))

In [ ]:
## Verify distribution with log transform 
X_full[numeric_features].hist(bins=100, figsize=(24,12), log = True)

The above distribution looks good after log transformation

In [ ]:
## Box Plot for Outliers
fig = plt.figure(figsize=(18,6))
sns.boxplot(data=X_full[numeric_features], orient="h", palette="Set2");
plt.xticks(fontsize= 14)
plt.title('Box plot of numerical columns', fontsize=16);

Looks like few outliers in Cont0, Cont6, Cont8, target columns.
Lets check the ouliers in  target column now.

# Step 3: Prepare the data



In [ ]:
# Deal with duplicate rows
X_full[my_features].drop_duplicates()

No Duplicate Rows in the data set...

In [ ]:
# Deal with Outliers

#remove outliers from target column 
#for colName in [['target']]:
    #X_full = treatoutliers(df=X_full,columns=colName, treatment='remove')         
    
#Quantile-based Flooring and Capping
for colName in [['target','cont0','cont6','cont8']]:
    X_full = treatoutliers(df=X_full,columns=colName, treatment='cap')      
    
ShowEssentialInfo(X_full)

In [ ]:
## Box Plot for Outliers
fig = plt.figure(figsize=(18,6))
sns.boxplot(data=X_full[numeric_features], orient="h", palette="Set2");
plt.xticks(fontsize= 14)
plt.title('Box plot of numerical columns after handling Outliers', fontsize=16);

In [ ]:
# Deal with missing data
## No Missing data in this dataset :)

The next code cell separates the target (which we assign to `y`) from the training features.

In [ ]:
if full_run == 'N' :
    X_random_subset = get_sample_dataset(X_full, categorical_features, max_rows_per_class)
else:
    X_random_subset = X_full.copy()
    
ShowEssentialInfo(X_random_subset)

In [ ]:
# Remove rows with missing target, separate target from predictors
X_random_subset.dropna(axis=0, subset=[target_feature], inplace=True)
y = X_random_subset.pop(target_feature)


#Prieview features
ShowEssentialInfo(X_random_subset)


In [ ]:
# A scatter plot matrix is a grid (or matrix) of scatter plots used to visualize bivariate relationships between combinations of variables. 
# Each scatter plot in the matrix visualizes the relationship between a pair of variables, allowing many relationships to be explored in one chart.

#pd.plotting.scatter_matrix(X_random_subset[:1000], figsize=(30, 20), alpha=0.2)



In [ ]:
# Break off validation set from training data
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X_random_subset, y, 
                                                                train_size=0.9, test_size=0.1,
                                                                random_state=0)

In [ ]:
# Identify Numeric / Categorical features this would be used to transform features later 

# "Cardinality" means the number of unique values in a column
# Select categorical columns with relatively low cardinality (convenient but arbitrary)
categorical_features  = [cname for cname in X_full.columns if
                    X_full[cname].dtype in ["object"] and cname not in exclude_columns
                        ]

# Select numerical columns
numeric_features  = [cname for cname in X_full.columns if 
                X_full[cname].dtype in ['int64', 'float64'] and
                     cname not in exclude_columns
                 ]

#remove target column from Numeric / categorical features
if numeric_features.count(target_feature) == 1:
    numeric_features.remove(target_feature)
elif  categorical_features.count(target_feature) == 1:
    categorical_features.remove(target_feature)

# Keep selected columns only
my_features = categorical_features + numeric_features

print('categorical_features({}):'.format(len(categorical_features)),categorical_features)
print('numeric_features({}):'.format(len(numeric_features)), numeric_features)
print('my_features:({})'.format(len(my_features)), my_features)

In [ ]:
X_train = X_train_full[my_features]
X_valid = X_valid_full[my_features]
X_test = X_test_full[my_features]

In [ ]:
ShowEssentialInfo(X_train)

In [ ]:
ShowEssentialInfo(X_valid)

In [ ]:
ShowEssentialInfo(X_test)

# Step 4: Train a model

Now that the data is prepared, the next step is to train a model.  

Lets fit a XG Boost Regression model to the data.

In [ ]:
# Define the model Parameters, can be optimized using either Optuna or Grid Search CV

#CPU parameters
#lgbm_params = {'n_estimators' : 10000,  'max_depth' : 2, 'learning_rate' : 0.1, 'subsample' : 0.95, 'colsample_bytree' : 0.85, 'reg_alpha' : 30.0, 'reg_lambda' : 25.0 , 'num_leaves' : 4, 'max_bin' : 512, 'random_state' : random_seed}
#xgb_params = {'n_estimators': 10000, 'max_depth': 3, 'learning_rate': 0.03628302216953097, 'gamma': 0, 'min_child_weight': 1, 'subsample': 0.7875490025178415, 'colsample_bytree': 0.11807135201147481, 'reg_alpha': 23.13181079976304, 'reg_lambda': 0.0008746338866473539, 'random_state':random_seed}

#GPU parameters
lgbm_params = {'n_estimators' : 10000, 'max_depth' : 2, 'learning_rate' : 0.1, 'subsample' : 0.95, 'colsample_bytree' : 0.85, 'reg_alpha' : 30.0, 'reg_lambda' : 25.0 , 'num_leaves' : 4, 'random_state' : random_seed, 'device':'gpu'}
xgb_params = {'n_estimators': 10000, 'max_depth': 3, 'learning_rate': 0.036, 'gamma': 0, 'min_child_weight': 1, 'subsample': 0.79, 'colsample_bytree': 0.112, 'reg_alpha': 23.132, 'reg_lambda': 0.0009, 'random_state':random_seed, 'tree_method':'gpu_hist', 'predictor':'gpu_predictor'}

model = XGBRegressor(**xgb_params) 
lgbm_model = LGBMRegressor(**lgbm_params)

#model = Ridge(alpha=0.05, normalize=True) #  RMSE: 0.7410128061594929
#model = Lasso(alpha=0.5, normalize=True)  # RMSE: 0.747933770739031
#model = LinearRegression(normalize=True)  # RMSE: 0.7410032228563359
#model = DecisionTreeRegressor(max_depth=3) #RMSE: 0.7429294772568634
#model = RandomForestRegressor(n_estimators=50, random_state=rans, max_depth=3) #RMSE: 0.7421769885950228
#model = XGBRegressor(n_estimators=500, learning_rate=0.35, n_jobs=-1, random_state=rans, eval_metric ='rmse', objective ='reg:squarederror', booster='gblinear') 
#RMSE: 0.748752632103789
#model = BaggingRegressor(RandomForestRegressor(n_estimators=50, random_state=rans, max_depth=3), n_estimators=2, random_state=rans, max_samples=0.8, max_features=0.7, bootstrap=True, bootstrap_features=True, n_jobs=-1)  
#RMSE: 0.742235730626387
#model = MLPRegressor(activation='tanh', hidden_layer_sizes= (100,3) ,learning_rate='adaptive', solver='adam', max_iter=1000)
#RMSE: 0.7342802036691075




In [ ]:
%%time

#transformer = FunctionTransformer(log_transform)

# Preprocessing for numerical data
numerical_transformer = Pipeline(steps=[
       ('imputer', SimpleImputer(strategy='mean'))
       #,('transformer', transformer)
       ,('RobustScaler', RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25.0, 75.0), copy=True))  
       ,('scaler', StandardScaler())
      # ,('scaler', MinMaxScaler())
      #,('normalizer',  Normalizer())
])

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    #('imputer', SimpleImputer(strategy='constant'))
    ('imputer', SimpleImputer(strategy='most_frequent')) 
    #,('onehot', OneHotEncoder(handle_unknown='ignore'))
    ,('scaler', OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])


# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder="passthrough"
  )


In [ ]:
#pca = PCA(n_components=50)
#Bundle preprocessing and modeling code in a pipeline
xgb_bundle = Pipeline(steps=[('preprocessor', preprocessor),
                      #('pca',pca),
                      ('model', model)
                     ])


lgbm_bundle = Pipeline(steps=[('preprocessor', preprocessor),
                      #('pca',pca),
                      ('model', lgbm_model)
                     ])

## TransformedTargetRegressor pipeline Scales Target column during training and inverts the transformations during prediction
xgbclf = TransformedTargetRegressor(regressor=xgb_bundle, transformer=StandardScaler())
lgbmclf = TransformedTargetRegressor(regressor=lgbm_bundle, transformer=StandardScaler())

#xgbclf = xgb_bundle
#lgbmclf = lgbm_bundle

In [ ]:
## XGB Regressor
from sklearn import set_config
set_config(display='diagram')
xgbclf


In [ ]:
## LGBM Regressor,
lgbmclf

In [ ]:
nSplits = 10
kf = KFold(n_splits=nSplits, shuffle=True, random_state=random_seed)

In [ ]:
%%time


#final_model = clf.fit(X_train, y_train)    
#preds_valid = final_model.predict(X_valid)
#print MAE, RMSE
#print('MAE:',mean_absolute_error(y_valid, preds_valid))
#print('RMSE:',mean_squared_error(y_valid, preds_valid, squared=False))
    

avg_valid_rmse = 0 # initialise variable for the average RMSE 
preds_valid = 0 # initialise variable for the Validation set predicitons 
predictions = 0 # initialise variable for the Test set predicitons
ncount =1


#for train_idx, test_idx in kf.split(X_random_subset.iloc[rand_idx]):
for train_idx, test_idx in kf.split(X_random_subset[my_features]):    
    
    X_train_split, X_valid_split = X_random_subset[my_features].iloc[train_idx], X_random_subset[my_features].iloc[test_idx]
    y_train_split, y_valid_split = y.iloc[train_idx], y.iloc[test_idx]
    
    #if ncount%2==0:
    if ncount%2==0:
        print('XGB')
        final_model = xgbclf.fit(X_train_split, y_train_split)
    else :
        print('XGB')
        final_model = xgbclf.fit(X_train_split, y_train_split)
        
    
    #Predict current Validation fold using the fold model
    preds_valid_split = final_model.predict(X_valid_split)
    
    #Calculate the rmse error for current Validation fold
    rmse = mean_squared_error(preds_valid_split, y_valid_split, squared=False)
    print("rmse:{}".format(rmse))
    
    #Average of the rmse errors
    avg_valid_rmse += rmse / nSplits
    
    #Predict Total Validation & Test sets using the fold Model 
    preds_valid_all = final_model.predict(X_valid)
    preds_test_all = final_model.predict(X_test)

    #Average of the predictions
    preds_valid += preds_valid_all / nSplits
    predictions += preds_test_all / nSplits
    ncount += 1

print("Average Validation rmse: {}".format(avg_valid_rmse))   



In [ ]:
result_df=pd.DataFrame({'Actual':y_valid, 'Predicted':preds_valid, 'Diff':preds_valid-y_valid})  
result_df['Diff'].round().value_counts()

In [ ]:
## setting plot style
plt.style.use('fivethirtyeight')
  
## plotting residual errors in training data
plt.scatter(final_model.predict(X_train), final_model.predict(X_train) - y_train,
            color = "green", s = 10, label = 'Train data')
  
## plotting residual errors in Validation data
plt.scatter(preds_valid, preds_valid-y_valid,
            color = "blue", s = 10, label = 'Validation data')
  
## plotting line for zero residual error
plt.hlines(y = 0, xmin = 0, xmax = 50, linewidth = 2)
  
## plotting legend
plt.legend(loc = 'upper right')
  
## plot title
plt.title("Residual errors")
  
## method call for showing the plot
plt.show()

In [ ]:
final_model.get_params

In [ ]:
#Compare results
plt.plot(y_valid.values, label='Actual')
plt.plot(preds_valid, label='Predicted')
plt.ylabel('Target')

plt.legend()
plt.show()


# Step 5: Submit to the competition

We'll begin by using the trained model to generate predictions, which we'll save to a CSV file.

In [ ]:
# Use the model to generate predictions
#predictions = final_model.predict(X_test)

# Save the predictions to a CSV file
output = pd.DataFrame({'Id': X_test.index,
                       'target': predictions})
output.to_csv('submission.csv', index=False)